<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z333_ZscoreQuantile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Z-score & Quantile Regression

Dos variantes nuevas de predicción:

### 1. Regresión lineal sobre z-score (`reg_zscore`)
Normaliza la serie por `(x - media) / std` antes de ajustar la recta, y luego desnormaliza la predicción.
- Ventaja: elimina diferencias de escala entre productos (un producto de 1 tn y otro de 1000 tn pesan igual en el ajuste)
- Permite comparar pendientes entre productos en términos relativos

### 2. Regresión por cuantil (`quantile_reg`)
En vez de minimizar el error cuadrático (media), minimiza el error absoluto ponderado → estima la **mediana** (q=0.5) o cualquier otro cuantil.
- Ventaja: robusta a outliers (un mes de venta anómala no rompe la predicción)
- `QuantileRegressor(quantile=0.5)` es equivalente a regresión de mínimos absolutos
- Probamos q=0.3 (predicción conservadora) y q=0.5

### Backtesting
Train hasta 201910, target 201912 (mismo split que z330).

## 0. Init Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "product_id_apredecir201912.txt"

# 1. Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle

In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from scipy.stats import chi2, runstest_1samp
from sklearn.linear_model import LinearRegression, QuantileRegressor

import warnings
warnings.filterwarnings('ignore')

In [ ]:
PARAM = {
    'experimento':        'ZscoreQuantile-01',
    'kaggle_competition': 'labo-iii-2026-rosario',
    'ventana':            6,    # ventana en meses para todos los modelos
    'cuantil':            0.5,  # 0.5 = mediana, 0.3 = conservador
    'periodo_corte':      201910,
    'periodo_target':     201912,
}

ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2. Datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")
tb_ventas    = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])

tb_train = tb_ventas.filter(pl.col("periodo") <= PARAM['periodo_corte'])
tb_real  = (
    tb_ventas
    .filter(pl.col("periodo") == PARAM['periodo_target'])
    .select(["product_id", "tn"])
    .rename({"tn": "tn_real"})
)

productos = tb_apredecir["product_id"].to_list()
print(f"{len(productos)} productos")

# 3. Funciones de predicción

Tres funciones, todas con la misma firma: `(serie, ventana, horizonte) → float`

- `pred_ols`: regresión OLS clásica (baseline, igual que z330)
- `pred_zscore`: OLS sobre z-score normalizado, desnormaliza al final
- `pred_quantile`: regresión de cuantil (q=0.5 por defecto → mediana)

In [ ]:
def pred_ols(serie: np.ndarray, ventana: int, horizonte: int = 2) -> float:
    w = min(ventana, len(serie))
    if w < 2:
        return max(float(serie.mean()), 0.0)
    y = serie[-w:]
    x = np.arange(w).reshape(-1, 1)
    pred = float(LinearRegression().fit(x, y).predict([[w - 1 + horizonte]])[0])
    return max(pred, 0.0)


def pred_zscore(serie: np.ndarray, ventana: int, horizonte: int = 2) -> float:
    """
    Normaliza los últimos `ventana` puntos por su media y std,
    ajusta OLS en ese espacio y desnormaliza la predicción.
    Útil cuando la escala de la serie cambia mucho a lo largo del tiempo.
    """
    w = min(ventana, len(serie))
    if w < 2:
        return max(float(serie.mean()), 0.0)

    y = serie[-w:].copy()
    mu  = y.mean()
    std = y.std() + 1e-9

    y_norm = (y - mu) / std
    x = np.arange(w).reshape(-1, 1)

    pred_norm = float(LinearRegression().fit(x, y_norm).predict([[w - 1 + horizonte]])[0])
    pred = pred_norm * std + mu  # desnormalizar
    return max(pred, 0.0)


def pred_quantile(serie: np.ndarray, ventana: int, horizonte: int = 2,
                  q: float = 0.5) -> float:
    """
    Regresión de cuantil sobre los últimos `ventana` puntos.
    q=0.5 → mediana (mínimos absolutos), robusta a outliers.
    q=0.3 → predicción conservadora (útil para productos en caída).
    """
    w = min(ventana, len(serie))
    if w < 2:
        return max(float(np.median(serie)), 0.0)

    y = serie[-w:]
    x = np.arange(w).reshape(-1, 1)

    # alpha=0 → sin regularización (solo cuantil puro)
    pred = float(QuantileRegressor(quantile=q, alpha=0).fit(x, y).predict([[w - 1 + horizonte]])[0])
    return max(pred, 0.0)

# 4. Backtesting — comparación de variantes

In [ ]:
ventana = PARAM['ventana']
q       = PARAM['cuantil']

resultados = []

for pid in productos:
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    # naive: mediana últimos 6 meses
    naive = max(float(np.median(serie[-6:])), 0.0)

    resultados.append({
        'product_id':    pid,
        'pred_naive':    naive,
        'pred_ols':      pred_ols(serie,      ventana, horizonte=2),
        'pred_zscore':   pred_zscore(serie,   ventana, horizonte=2),
        'pred_q50':      pred_quantile(serie, ventana, horizonte=2, q=0.5),
        'pred_q30':      pred_quantile(serie, ventana, horizonte=2, q=0.3),
    })

tb_preds = pl.DataFrame(resultados)
print("Predicciones listas")

In [ ]:
tb_bt = tb_real.join(tb_preds, on='product_id', how='left')

modelos = ['naive', 'ols', 'zscore', 'q50', 'q30']
for m in modelos:
    tb_bt = tb_bt.with_columns(
        (pl.col('tn_real') - pl.col(f'pred_{m}')).abs().alias(f'err_{m}')
    )

print(f"RMSE — backtesting 201912  (ventana={ventana}m)")
print()
rmse_base = None
for m in modelos:
    rmse = float(np.sqrt((tb_bt[f'err_{m}'] ** 2).mean()))
    if m == 'naive':
        rmse_base = rmse
        print(f"  {'naive':10s}: {rmse:.4f}")
    else:
        delta = rmse - rmse_base
        print(f"  {m:10s}: {rmse:.4f}  ({delta:+.4f} vs naive)")

# 5. ¿OLS vs z-score — en qué se diferencian?

La normalización z-score solo cambia el resultado cuando la media de la ventana es muy distinta al std.
Graficamos la diferencia entre ambas predicciones para entender cuándo importa.

In [ ]:
diff = (tb_bt['pred_ols'] - tb_bt['pred_zscore']).to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(diff, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_xlabel('OLS pred − z-score pred')
axes[0].set_ylabel('productos')
axes[0].set_title(f'Diferencia de predicciones OLS vs z-score\nmedia={diff.mean():.3f}, std={diff.std():.3f}')

# scatter: pred_ols vs pred_zscore
axes[1].scatter(tb_bt['pred_ols'].to_numpy(), tb_bt['pred_zscore'].to_numpy(),
                s=8, alpha=0.5, color='steelblue')
lim = max(tb_bt['pred_ols'].max(), tb_bt['pred_zscore'].max())
axes[1].plot([0, lim], [0, lim], 'r--', linewidth=0.8)
axes[1].set_xlabel('pred OLS')
axes[1].set_ylabel('pred z-score')
axes[1].set_title('OLS vs z-score — predicciones por producto')

plt.tight_layout()
plt.show()

# 6. OLS vs Quantile — qué pasa con series que tienen outliers

La regresión por cuantil (q50) debería ser más conservadora cuando hay meses anómalos.
Buscamos los productos donde más difieren.

In [ ]:
tb_bt = tb_bt.with_columns(
    (pl.col('pred_ols') - pl.col('pred_q50')).alias('diff_ols_q50')
)

# productos donde OLS predice mucho más que q50 → outlier influyendo al alza
casos_ols_alto = (
    tb_bt.sort('diff_ols_q50', descending=True).head(6)['product_id'].to_list()
)
# productos donde q50 predice más que OLS → outlier influyendo a la baja en OLS
casos_q50_alto = (
    tb_bt.sort('diff_ols_q50').head(6)['product_id'].to_list()
)

def plot_comparacion(pids, titulo):
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.flatten()
    for i, pid in enumerate(pids):
        serie_full = tb_ventas.filter(pl.col('product_id') == pid).sort('periodo')
        periodos_  = serie_full['periodo'].to_list()
        tn_        = serie_full['tn'].to_numpy().astype(float)

        idx_corte  = next((j for j, p in enumerate(periodos_) if p > PARAM['periodo_corte']), len(periodos_))
        idx_target = next((j for j, p in enumerate(periodos_) if p == PARAM['periodo_target']), None)
        tn_train_  = tn_[:idx_corte]

        row = tb_bt.filter(pl.col('product_id') == pid)
        real_val = float(row['tn_real'][0])
        ols_val  = float(row['pred_ols'][0])
        q50_val  = float(row['pred_q50'][0])
        q30_val  = float(row['pred_q30'][0])
        zsc_val  = float(row['pred_zscore'][0])

        ax = axes[i]
        ax.plot(range(len(tn_train_)), tn_train_, 'o-', color='steelblue',
                markersize=3, linewidth=1.5, label='historia')

        if idx_target is not None:
            t = idx_target
            ax.scatter([t], [real_val], color='black',   s=90,  zorder=6, label=f'real={real_val:.1f}')
            ax.scatter([t], [ols_val],  color='tomato',  s=60,  zorder=5, marker='D', label=f'OLS={ols_val:.1f}')
            ax.scatter([t], [q50_val],  color='purple',  s=60,  zorder=5, marker='s', label=f'q50={q50_val:.1f}')
            ax.scatter([t], [q30_val],  color='orange',  s=60,  zorder=5, marker='^', label=f'q30={q30_val:.1f}')
            ax.scatter([t], [zsc_val],  color='green',   s=40,  zorder=5, marker='v', label=f'zsc={zsc_val:.1f}')

        ax.set_title(f'pid {pid}', fontsize=8)
        ax.legend(fontsize=5)

    fig.suptitle(titulo, fontsize=10)
    plt.tight_layout()
    plt.show()

plot_comparacion(casos_ols_alto, 'OLS predice MUCHO MÁS que q50 → outlier infló OLS')
plot_comparacion(casos_q50_alto, 'q50 predice más que OLS → outlier deprimió OLS')

# 7. McNemar — significancia estadística

In [ ]:
def mcnemar(err_a, err_b, nombre_a, nombre_b):
    n10 = (err_a < err_b).sum()
    n01 = (err_b < err_a).sum()
    if n10 + n01 == 0:
        print(f"{nombre_a} vs {nombre_b}: sin discrepancias")
        return
    chi2_stat = (abs(n10 - n01) - 1)**2 / (n10 + n01)
    pvalue    = 1 - chi2.cdf(chi2_stat, df=1)
    ganador   = nombre_a if n10 > n01 else nombre_b
    sig = "** SIG **" if pvalue < 0.05 else "no sig   "
    print(f"{nombre_a:10s} vs {nombre_b:10s}:  {nombre_a} gana {n10:3d} | {nombre_b} gana {n01:3d}  p={pvalue:.4f}  {sig}  → {ganador}")

print("McNemar — 780 productos:")
print()
err_naive = tb_bt['err_naive'].to_numpy()
for m in ['ols', 'zscore', 'q50', 'q30']:
    mcnemar(err_naive, tb_bt[f'err_{m}'].to_numpy(), 'naive', m)

print()
# ols vs variantes
err_ols = tb_bt['err_ols'].to_numpy()
for m in ['zscore', 'q50', 'q30']:
    mcnemar(err_ols, tb_bt[f'err_{m}'].to_numpy(), 'ols', m)

# 8. Distribución de errores — OLS vs q50

El RMSE penaliza los errores grandes. Graficamos la distribución de errores
para ver si q50 recorta la cola derecha (errores grandes).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# histograma de errores absolutos
e_ols = tb_bt['err_ols'].to_numpy()
e_q50 = tb_bt['err_q50'].to_numpy()
e_naive = tb_bt['err_naive'].to_numpy()

cap = np.percentile(np.concatenate([e_ols, e_q50, e_naive]), 95)
bins = np.linspace(0, cap, 50)

axes[0].hist(e_naive, bins=bins, alpha=0.4, label='naive',  color='gray')
axes[0].hist(e_ols,   bins=bins, alpha=0.5, label='OLS',   color='tomato')
axes[0].hist(e_q50,   bins=bins, alpha=0.5, label='q50',   color='purple')
axes[0].set_xlabel('error absoluto')
axes[0].set_ylabel('productos')
axes[0].set_title('Distribución de errores absolutos (hasta p95)')
axes[0].legend()

# scatter: err_ols vs err_q50
axes[1].scatter(e_ols, e_q50, s=8, alpha=0.4, color='purple')
lim = max(e_ols.max(), e_q50.max())
axes[1].plot([0, lim], [0, lim], 'r--', linewidth=0.8)
axes[1].set_xlabel('err OLS')
axes[1].set_ylabel('err q50')
q50_gana = (e_q50 < e_ols).sum()
axes[1].set_title(f'OLS vs q50 por producto\nq50 gana en {q50_gana}/{len(e_ols)}')

plt.tight_layout()
plt.show()

# 9. z-score como feature de selección de modelo

El z-score del último valor de la serie (`z_last = (tn_last - media) / std`) indica si el producto
está en un valor anómalo respecto a su historia reciente:
- `z_last >> 0`: mes anómalamente alto → q50 o q30 probablemente mejor (no extrapolar ese nivel)
- `z_last << 0`: mes anómalamente bajo → idem
- `z_last ≈ 0`: en su media habitual → OLS funciona bien

Analizamos si esta métrica predice cuándo q50 gana a OLS.

In [ ]:
z_lasts = []
for pid in productos:
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    w = min(PARAM['ventana'], len(serie))
    ult = serie[-w:]
    z = (serie[-1] - ult.mean()) / (ult.std() + 1e-9)
    z_lasts.append({'product_id': pid, 'z_last': float(z)})

tb_bt = tb_bt.join(pl.DataFrame(z_lasts), on='product_id', how='left')

# ¿q50 gana más cuando |z_last| es alto?
tb_bt = tb_bt.with_columns(
    pl.col('z_last').abs().alias('abs_z_last'),
    (pl.col('err_ols') - pl.col('err_q50')).alias('mejora_q50')
)

z_abs = tb_bt['abs_z_last'].to_numpy()
mejora = tb_bt['mejora_q50'].to_numpy()

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(z_abs, mejora, s=10, alpha=0.4,
                c=mejora, cmap='RdYlGn', vmin=-20, vmax=20)
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('|z_last|  (anomalía del último mes respecto a ventana)')
ax.set_ylabel('mejora q50 vs OLS (> 0 → q50 gana)')
ax.set_title('z-score del último mes vs ventaja de q50')
plt.colorbar(sc, ax=ax, label='mejora q50')
plt.tight_layout()
plt.show()

# correlación
corr = float(np.corrcoef(z_abs, mejora)[0, 1])
print(f"Correlación |z_last| ~ mejora_q50: {corr:.4f}")

# cuantificar: q50 gana más cuando z_last es alto
umbral = np.median(z_abs)
mascara_alto = z_abs > umbral
print(f"\nCuando |z_last| > mediana ({umbral:.2f}):")
print(f"  q50 gana en {(mejora[mascara_alto] > 0).sum()} de {mascara_alto.sum()} productos")
print(f"Cuando |z_last| <= mediana:")
print(f"  q50 gana en {(mejora[~mascara_alto] > 0).sum()} de {(~mascara_alto).sum()} productos")

# 10. Submit — modelo elegido

Usamos toda la historia hasta 201912 para predecir 202002.

Variantes para probar en Kaggle:
- `'modelo': 'q50'` — cuantil mediana
- `'modelo': 'zscore'` — OLS sobre z-score normalizado
- `'modelo': 'q30'` — cuantil conservador
- `'modelo': 'adaptativo'` — q50 cuando |z_last| alto, OLS si no

In [ ]:
PARAM_SUBMIT = {
    'modelo':  'q50',       # 'q50', 'zscore', 'q30', 'adaptativo'
    'ventana': 6,
    'cuantil': 0.5,
    'z_umbral': 1.0,        # solo para modo 'adaptativo'
}

tb_full = tb_ventas
preds_final = []

for pid in productos:
    serie = (
        tb_full.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )

    v = PARAM_SUBMIT['ventana']
    modelo = PARAM_SUBMIT['modelo']

    if modelo == 'q50':
        pred = pred_quantile(serie, ventana=v, horizonte=2, q=0.5)

    elif modelo == 'zscore':
        pred = pred_zscore(serie, ventana=v, horizonte=2)

    elif modelo == 'q30':
        pred = pred_quantile(serie, ventana=v, horizonte=2, q=0.3)

    elif modelo == 'adaptativo':
        # usa q50 cuando el último valor es anómalo, OLS si está en su media
        ult = serie[-v:]
        z = abs(serie[-1] - ult.mean()) / (ult.std() + 1e-9)
        if z > PARAM_SUBMIT['z_umbral']:
            pred = pred_quantile(serie, ventana=v, horizonte=2, q=0.5)
        else:
            pred = pred_ols(serie, ventana=v, horizonte=2)

    else:
        pred = pred_ols(serie, ventana=v, horizonte=2)

    preds_final.append({'product_id': pid, 'tn': pred})

tb_final = pl.DataFrame(preds_final)
display(tb_final.head(10))
print(f"Nulls: {tb_final['tn'].is_null().sum()}")

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
    import os
    os.system(f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"')

modelo   = PARAM_SUBMIT['modelo']
ventana  = PARAM_SUBMIT['ventana']
archivo  = f"{modelo}_{ventana}m.csv"
mensaje  = f"Reg {modelo} ventana {ventana}m → t+2"

tb_final.write_csv(archivo)
kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje)